In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os, sys, inspect
from dotenv import load_dotenv

In [ ]:
from datasets import load_dataset
import wandb
import numpy as np

import torch
from torch.nn import functional as F
from transformers import AutoModelForMultimodalLM, AutoProcessor

In [ ]:
from PIL import Image
import requests
from io import BytesIO

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))

In [ ]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

### Step-1: - Setup Model 

In [ ]:
import torchao
from torchao.quantization import quantize_, Int8WeightOnlyConfig

In [ ]:
MODEL_ID = "thinkingmachines/TML-d24-varloop-research"
MAX_MEMORY = {i: "35GiB" for i in range(2)}

In [ ]:
def build_device_map(gpu0=0, gpu1=1):
    device_map = {
        "model.vision_tower": gpu0,
        "model.language_model.embed_tokens": gpu0,
        "model.language_model.norm": gpu0,
        "model.language_model.final_input": gpu0,
        "model.language_model.message_input": gpu1,
        "lm_head": gpu0,
        "logit_softcap": gpu0,
    }

    for i in range(24):
        device_map[f"model.language_model.layers.{i}"] = (
            gpu1 if 6 <= i < 18 else gpu0
        )

    return device_map

DEVICE_MAP = build_device_map()

In [ ]:
def load_model(R=3):
    assert 1 <= R <= 5
    # assert torch.cuda.device_count() >= 4
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model, loading_info = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map="balanced", max_memory=MAX_MEMORY, experts_implementation="grouped_mm",
                                                                   dtype=torch.bfloat16, text_config={"loop_num_passes": R}, output_loading_info=True)

    model.eval()
    model.requires_grad_(False)

    print("Loop count:", model.config.text_config.loop_num_passes)
    print("Device map:", model.hf_device_map)

    return model, processor, loading_info
model, processor, loading_info = load_model(R=1)

In [ ]:
def set_loops(R):
    assert 1 <= R <= 5
    model.config.text_config.loop_num_passes = R
    language_model.config.loop_num_passes = R


def prepare_inputs(content):
    return processor.apply_chat_template(
        [{"role": "user", "content": content}],
        reasoning_effort="none",
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


def synchronize():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

#### Step-2: - Understanding Model Architecture

In [ ]:
state_hf = model.state_dict()
for k,v in state_hf.items(): 
    print(k,v.shape)

In [ ]:
language_model = model.model.language_model
vision_tower = model.model.vision_tower

In [ ]:
print(model.config.vision_config)

In [ ]:
for name, module in vision_tower.named_children():
    print(name, type(module).__name__)

In [ ]:
print(inspect.getsource(type(vision_tower).forward))

In [ ]:
print(inspect.getsource(type(model.model).forward))

#### Sample Input 

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"

image = Image.open(
    BytesIO(requests.get(url, timeout=30).content)
).convert("RGB")

image.thumbnail((512, 512))

question = "What animal is on the candy?"

content = [
    {"type": "image", "image": image},
    {"type": "text", "text": question}
]

inputs = prepare_inputs(content)

for key, value in inputs.items():
    print(key, value.shape)

In [ ]:
print(inspect.getsource(type(model.model).get_image_features))

#### Extract projected visual tokens

In [ ]:
@torch.inference_mode()
def extract_visual_tokens(inputs):
    outputs = model.model.get_image_features(
        pixel_values=inputs["pixel_values"],
        image_grid_thw=inputs["image_grid_thw"],
        return_dict=True
    )

    visual_tokens = torch.cat(outputs.pooler_output, dim=0)

    return visual_tokens, outputs

visual_tokens, vision_outputs = extract_visual_tokens(inputs)

print("Image resolution:", image.size)
print("Image grid:", inputs["image_grid_thw"])
print("Visual tokens:", visual_tokens.shape)
print("Vision hidden states:", vision_outputs.last_hidden_state.shape)

In [ ]:
image_mask = inputs["input_ids"] == model.config.image_token_id
num_placeholders = image_mask.sum().item()

print("Image placeholders:", num_placeholders)
print("Projected visual tokens:", visual_tokens.shape[0])

assert num_placeholders == visual_tokens.shape[0]
assert visual_tokens.shape[-1] == 2304

### Step-4: - Token pruning 

In [ ]:
def random_selector(visual_tokens, k=0.5, seed=42):
    assert 0 < k <= 1

    N = visual_tokens.shape[0]
    n_keep = max(1, int(N * k))

    generator = torch.Generator().manual_seed(seed)

    indices = torch.randperm(N, generator=generator)
    indices = indices[:n_keep].sort().values

    return indices.to(visual_tokens.device)

In [ ]:
from copy import copy

def prune_inputs(inputs, vision_outputs, indices):
    input_ids = inputs["input_ids"]
    assert input_ids.shape[0] == 1
    assert len(vision_outputs.pooler_output) == 1

    image_mask = input_ids[0] == model.config.image_token_id
    image_positions = image_mask.nonzero(as_tuple=True)[0]

    visual_tokens = vision_outputs.pooler_output[0]

    assert len(image_positions) == len(visual_tokens)

    # Retain every non-image token
    keep_mask = ~image_mask

    # Retain selected image placeholders
    keep_mask[image_positions[indices.to(input_ids.device)]] = True

    # Retain corresponding visual embeddings
    pruned_vision = copy(vision_outputs)
    pruned_vision.pooler_output = (
        visual_tokens[indices.to(visual_tokens.device)],
    )

    return {
        "input_ids": input_ids[:, keep_mask],
        "attention_mask": inputs["attention_mask"][:, keep_mask],
        "mm_encoder_outputs": {"image": pruned_vision}
    }

In [ ]:
for k in [1.0, 0.5, 0.25]:
    indices = random_selector(visual_tokens, k)
    pruned = prune_inputs(inputs, vision_outputs, indices)

    print(
        f"k={k} | "
        f"Visual tokens={len(indices)} | "
        f"Total tokens={pruned['input_ids'].shape[-1]}"
    )

In [ ]:
set_loops(3)

with torch.inference_mode():
    original = model(
        **inputs,
        use_cache=False,
        logits_to_keep=1
    ).logits.float()

    indices = random_selector(visual_tokens, k=1.0)
    pruned = prune_inputs(inputs, vision_outputs, indices)

    reconstructed = model(
        **pruned,
        use_cache=False,
        logits_to_keep=1
    ).logits.float()

print("Max difference:",
      (original - reconstructed).abs().max().item())

print("Same predicted token:",
      torch.equal(
          original.argmax(-1),
          reconstructed.argmax(-1)
      ))

In [ ]:
@torch.inference_mode()
def generate_tokens(inputs, max_new_tokens=1024):
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True
    )

    return outputs[0, inputs["input_ids"].shape[-1]:]

In [ ]:
set_loops(3)

original_tokens = generate_tokens(inputs)

indices = random_selector(visual_tokens, k=1.0)
pruned = prune_inputs(inputs, vision_outputs, indices)

reconstructed_tokens = generate_tokens(pruned)

print("Identical generation:", torch.equal(original_tokens, reconstructed_tokens))

In [ ]:
for k in [1.0, 0.5, 0.25]:
    indices = random_selector(visual_tokens, k=k)
    pruned = prune_inputs(inputs, vision_outputs, indices)

    tokens = generate_tokens(pruned)
    answer = processor.decode(tokens, skip_special_tokens=True)

    print(f"\nRetention: {k * 100}%")
    print(f"Visual tokens: {len(indices)}")
    print(f"Decoder input tokens: {pruned['input_ids'].shape[-1]}")
    print(f"Output: {answer}")

In [ ]:
def decoder_hook(module, args, kwargs):
    print("Decoder input:", kwargs["inputs_embeds"].shape)

handle = language_model.register_forward_pre_hook(
    decoder_hook, with_kwargs=True
)

try:
    indices = random_selector(visual_tokens, k=0.5)
    pruned = prune_inputs(inputs, vision_outputs, indices)
    tokens = generate_tokens(pruned, max_new_tokens=3)
finally:
    handle.remove()

#### Uniform Pruning

In [ ]:
def uniform_selector(image_grid_thw, k=0.5):
    assert 0 < k <= 1

    t, h, w = image_grid_thw[0].tolist()
    assert t == 1

    N = h * w
    n_keep = max(1, int(N * k))

    if n_keep == N:
        return torch.arange(N)

    y, x = torch.meshgrid(
        torch.arange(h),
        torch.arange(w),
        indexing="ij"
    )

    coords = torch.stack([y, x], dim=-1)
    coords = coords.reshape(-1, 2).float()

    selected = []
    distances = torch.full((N,), float("inf"))
    idx = 0

    for _ in range(n_keep):
        selected.append(idx)

        d = ((coords - coords[idx]) ** 2).sum(-1)
        distances = torch.minimum(distances, d)
        distances[selected] = -1

        idx = distances.argmax().item()

    return torch.tensor(sorted(selected))

In [ ]:
set_loops(3)

for k in [1.0, 0.5, 0.25]:
    indices = uniform_selector(
        inputs["image_grid_thw"], k
    )

    pruned = prune_inputs(
        inputs, vision_outputs, indices
    )

    tokens = generate_tokens(pruned)

    print(f"\nRetention: {k * 100}%")
    print(f"Visual tokens: {len(indices)}")
    print(processor.decode(
        tokens, skip_special_tokens=True
    ))

#### Training-free text-guided pruning

In [ ]:
@torch.inference_mode()
def get_question_embeddings(question):
    embedding_layer = model.model.get_input_embeddings()

    text_ids = processor.tokenizer(
        question,
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    text_ids = text_ids.to(embedding_layer.weight.device)

    return embedding_layer(text_ids).squeeze(0)

text_embeddings = get_question_embeddings(question)

print("Text:", text_embeddings.shape)
print("Visual:", visual_tokens.shape)

In [ ]:
def text_guided_selector(visual_tokens, text_embeddings, k=0.5):
    v = F.normalize(visual_tokens.float(), dim=-1)
    q = text_embeddings.float().to(v.device)
    q = F.normalize(q.mean(dim=0), dim=-1)

    scores = v @ q

    n_keep = max(1, int(len(v) * k))
    indices = scores.topk(n_keep).indices.sort().values

    return indices, scores

In [ ]:
set_loops(3)

for k in [1.0, 0.5, 0.25]:
    indices, scores = text_guided_selector(visual_tokens, text_embeddings, k)
    pruned = prune_inputs(inputs, vision_outputs, indices)
    tokens = generate_tokens(pruned)
    print(f"\nRetention: {k * 100}%")
    print("Visual tokens:", len(indices))
    print(processor.decode(tokens, skip_special_tokens=True))

In [ ]:
import matplotlib.pyplot as plt
_, h, w = inputs["image_grid_thw"][0].tolist()

plt.imshow(scores.reshape(h, w).detach().cpu(),cmap="viridis")
plt.colorbar(label="Cosine similarity")
plt.title(question)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_relevance(image, question, scores, image_grid_thw):
    _, h, w = image_grid_thw[0].tolist()

    heatmap = scores.detach().float().cpu().reshape(h, w).numpy()

    # Match the processed image grid
    img = image.resize((w * 40, h * 40))

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(img)
    axes[0].set_title("Original image")

    axes[1].imshow(heatmap, cmap="jet", interpolation="nearest")
    axes[1].set_title(f"Cosine similarity ({h} × {w})")

    axes[2].imshow(img)
    overlay = axes[2].imshow(
        heatmap,
        cmap="jet",
        alpha=0.45,
        interpolation="nearest",
        extent=(0, img.width, img.height, 0)
    )
    axes[2].set_title("Question relevance overlay")

    for ax in axes:
        ax.axis("off")

    fig.colorbar(overlay, ax=axes, shrink=0.7,
                 label="Cosine similarity")

    fig.suptitle(question, fontsize=14)
    plt.show()

In [ ]:
questions = [
    "What animal is on the candy?",
    "animal",
    "orange candy",
    "green candy",
    "hand",
    "background"
]

for q in questions:
    text_emb = get_question_embeddings(q)

    _, scores = text_guided_selector(
        visual_tokens, text_emb, k=0.5
    )

    visualize_relevance(
        image, q, scores, inputs["image_grid_thw"]
    )

In [ ]:
attention = language_model.layers[0].self_attn

print(inspect.getsource(type(attention).forward))
print("Attention backend:",
      language_model.config._attn_implementation)

In [ ]:
set_loops(1)

with torch.inference_mode():
    outputs = model(
        **inputs,
        output_attentions=True,
        use_cache=False,
        logits_to_keep=1
    )

attentions = outputs.attentions

if attentions is None:
    print("Attention weights unavailable")
else:
    for i, attn in enumerate(attentions):
        print(i, attn.shape if attn is not None else None)

#### Decoder based pruning

In [ ]:
image_positions = (
    inputs["input_ids"][0] == model.config.image_token_id
).nonzero(as_tuple=True)[0]

ids = inputs["input_ids"][0].tolist()

q_ids = processor.tokenizer(
    question, add_special_tokens=False
)["input_ids"]

matches = [
    i for i in range(len(ids) - len(q_ids) + 1)
    if ids[i:i + len(q_ids)] == q_ids
    and i > image_positions[-1].item()
]

assert matches, "Question token sequence not found"

start = matches[-1]
question_positions = torch.arange(start, start + len(q_ids))

print("Image tokens:", len(image_positions))
print("Question tokens:", len(question_positions))
print("Question:", processor.decode(ids[start:start + len(q_ids)]))

In [ ]:
def attention_relevance(attention, question_positions, image_positions):
    A = attention[0].float()

    A = A.index_select(1, question_positions.to(A.device))
    A = A.index_select(2, image_positions.to(A.device))

    # Average across attention heads and question tokens
    scores = A.mean(dim=(0, 1))

    return scores

In [ ]:
for layer in [0, 1, 3, 5]:
    scores = attention_relevance(
        attentions[layer],
        question_positions,
        image_positions
    )

    visualize_relevance(
        image,
        f"Layer {layer}: {question}",
        scores,
        inputs["image_grid_thw"]
    )

In [ ]:

set_loops(3)
def attention_selector(scores, k=0.5):
    assert 0 < k <= 1

    n_keep = max(1, int(len(scores) * k))

    indices = scores.topk(n_keep).indices
    return indices.sort().values

for layer in range(6):
    scores = attention_relevance(
        attentions[layer],
        question_positions,
        image_positions
    )

    indices = attention_selector(scores, k=0.5)
    pruned = prune_inputs(inputs, vision_outputs, indices)

    tokens = generate_tokens(pruned)

    print(f"\nLayer: {layer}")
    print(processor.decode(tokens, skip_special_tokens=True))
